In [1]:
from dotenv import load_dotenv
from pprint import pprint
load_dotenv()

True

In [3]:
from langchain.chat_models import init_chat_model

**INITCHATMODEL SYNTAX**
###### init_chat_model(
######  model: str | None = None,
######  *,
######  model_provider: str | None = None,
######  configurable_fields: Literal['any'] | list[str] | tuple[str, ...] | None = None,
######  config_prefix: str | None = None,
######  **kwargs: Any = {}
###### -> BaseChatModel | _ConfigurableModel

**link:** "https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model"

In [25]:
chat = init_chat_model(model="groq:openai/gpt-oss-20b",
                       temperature =0.5
                       )

NameError: name 'init_chat_model' is not defined

In [ ]:
response = chat.invoke("Who are you?")
pprint(response.content_blocks)

[{'reasoning': 'We need to respond to "Who are you?" as ChatGPT. We should be '
               "friendly, mention that I'm an AI language model. Provide a "
               'brief explanation.',
  'type': 'reasoning'},
 {'text': 'I’m ChatGPT, a conversational AI developed by OpenAI. I’m built on '
          'the GPT‑4 architecture and designed to understand and generate '
          'human‑like text. I can help answer questions, offer explanations, '
          'brainstorm ideas, or just chat about a wide range of topics. If you '
          'have any specific needs or curiosities, just let me know!',
  'type': 'text'}]


**#CREATE AGENT**

**#Link:** "https://reference.langchain.com/python/langchain/agents/factory/create_agent"

     create_agent(
    model: str | BaseChatModel,
    tools: Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None = None,
    *,
    system_prompt: str | SystemMessage | None = None,
    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),
    response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None,
    state_schema: type[AgentState[ResponseT]] | None = None,
    context_schema: type[ContextT] | None = None,
    checkpointer: Checkpointer | None = None,
    store: BaseStore | None = None,
    interrupt_before: list[str] | None = None,
    interrupt_after: list[str] | None = None,
    debug: bool = False,
    name: str | None = None,
    cache: BaseCache[Any] | None = None,
    transformers: Sequence[TransformerFactory] | None = None
    ) -> CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]] 


In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from uuid import uuid7

In [ ]:
system_prompt = """You are mosquito bhat, a multistar michelin and a famous chef in India. You know A to Z in cooking. Even with a mosquito egg you will cook biriyani. Your job is to give cooking reciepe suggestions based on the ingredients user says. Your tone should be polite and witty. Make your answers grounded and don't hallucinate.Keep your response within 500 tokens"""

In [ ]:
model = create_agent(model="groq:openai/gpt-oss-20b", 
                     system_prompt=system_prompt,
                     checkpointer= InMemorySaver()
                     )
thread_id = str(uuid7())

In [ ]:
Question_ = "last time what did you respond??biriyani ? or tamarid ?"

In [ ]:
Question = HumanMessage(content=Question_)
config = {"configurable":{"thread_id":thread_id}}
response = model.invoke({"messages":[Question]},config)

In [ ]:
pprint(response['messages'][1].content_blocks)

**#Tool Usage**

In [ ]:
from tavily import TavilyClient
tavily_api = TavilyClient()
from langchain.tools import tool

In [36]:
@tool
def get_weather(prompt):
    """ Use this to fetch weather data from web"""
    return tavily_api.search(prompt)

In [37]:
system_prompt = """You are mosquito bhat, a multistar michelin and a famous chef in India and also a weatherman. You know A to Z in cooking and weatherman. Even with a mosquito egg you will cook biriyani. Your job is to give cooking reciepe suggestions based on the ingredients user says. Your tone should be polite and witty. Make your answers grounded and don't hallucinate.Keep your response within 500 tokens"""

In [38]:
model_tool = create_agent(model="groq:openai/gpt-oss-20b", 
                     system_prompt=system_prompt,
                     tools = [get_weather],
                     checkpointer= InMemorySaver()
                     )
thread_id = str(uuid7())

In [39]:
Question_ = "weather in chennai today?"

In [40]:
Question = HumanMessage(content=Question_)
config = {"configurable":{"thread_id":thread_id}}
response = model_tool.invoke({"messages":[Question]},config)

In [41]:
pprint(response['messages'][1].content_blocks)

[{'reasoning': 'We need to fetch weather. Use get_weather.',
  'type': 'reasoning'},
 {'args': {'prompt': 'weather in chennai today'},
  'id': 'fc_781c61ff-3532-4dff-9fad-1e356f78134f',
  'name': 'get_weather',
  'type': 'tool_call'}]


**# Tool Message**

In [42]:

pprint(response['messages'][2].content_blocks)

[{'text': '{"query": "weather in chennai today", "follow_up_questions": null, '
          '"answer": null, "images": [], "results": [{"url": '
          '"https://www.indiatoday.in/weather/chennai-weather-forecast-today", '
          '"title": "Chennai Weather Today (Sunday, Aug 16, 2026)", "content": '
          '"The minimum temperature in Chennai today is likely to hover around '
          '28 degrees Celsius, while the maximum temperature might reach 36 '
          'degrees Celsius. The mercury level is expected to hover around 29 '
          'degrees Celsius throughout the day, with the wind speed around '
          '3.89. The wind will move around 245 degrees with a gust speed of '
          '7.5. The sunrise time is 05:56 AM, while it will set at 06:30 PM on '
          'Sunday. As per the seven-day weather prediction, the temperature in '
          'Chennai is likely to reach 36 degrees [...] ### Download '
          'App\\n\\nAdvertisement\\n\\nAQI\\n\\nWeather\\n\\nNews / Wea

**# AI Response from the tool usage**

In [43]:
pprint(response['messages'][3].content_blocks)

[{'reasoning': 'We need to answer: "weather in chennai today?" Provide weather '
               'info. According to data: today Aug 16 2026, min 28, max 36, '
               'overcast, light drizzle, humidity 80-83%, feels like 32. '
               'Provide succinct answer within 500 tokens. Also we are '
               'chef/meteorologist. But question is only weather. Provide '
               'answer politely.',
  'type': 'reasoning'},
 {'text': '🌤️ **Chennai Weather – Today (Sunday,\u202f16\u202fAug\u202f'
          '2026)**  \n'
          '\n'
          '- **Temperature:** 28\u202f°C (min) – 36\u202f°C (max)  \n'
          '- **Feels‑like:** ~32\u202f°C (warm and humid)  \n'
          '- **Sky:** Overcast throughout the day with a chance of light '
          'drizzle in the afternoon  \n'
          '- **Humidity:** 80–83\u202f%  \n'
          '- **Wind:** 3–4\u202fkm/h from the south‑west, gusts up to 7–8\u202f'
          'km/h  \n'
          '- **Air Quality:** Good (AQI\u202f≈\u2